#Análise de Demonstrações Financeiras (CVM 2024) – Insights de Negócio
Este notebook consome os dados processados pelo pipeline ETL para realizar uma análise exploratória e estratégica das empresas listadas na CVM. O foco aqui é transformar os indicadores calculados (ROE, ROA, Margem Líquida) em inteligência de mercado através de consultas SQL.

#1. Conexão e Verificação do Database
Nesta etapa, estabelecemos a conexão com o banco de dados SQLite gerado pelo pipeline ETL e verificamos a integridade das tabelas antes de iniciar as consultas.

In [1]:
import pandas as pd
import logging
from sqlalchemy import create_engine, inspect, text

In [2]:
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("analise_cvm")

In [3]:
engine_cvm = create_engine('sqlite:///data_cvm.db')

def verificar_tabela():
  inspector = inspect(engine_cvm)
  tabelas = inspector.get_table_names()
  log.info(f"Tabelas no banco: {tabelas}")
  return tabelas

def query_sql(query_string):
  try:
    with engine_cvm.connect() as con:
      consulta = text(query_string)
      result = pd.read_sql(consulta, con=con)
      log.info(f"Query executada com sucesso ({len(result)} linhas).")
      return result
  except Exception as e:
    log.error(f"Erro ao executar a consulta: {str(e)}")
    return None

In [4]:
verificar_tabela()

['empresas_cvm']

In [5]:
df = query_sql("SELECT * FROM empresas_cvm")

df.head()

,CNPJ_CIA,DENOM_CIA,DT_REFER,RECEITA_VENDA,LUCRO_LÍQUIDO,ATIVO_TOTAL,PASSIVO_TOTAL,MARGEM_LÍQUIDA,ROA,PATRIMÔNIO_LÍQUIDO,ROE,PORTE_EMPRESA
0,00.000.000/0001-91,BCO BRASIL S.A.,2024-12-31 00:00:00.000000,2.735053e+11,2.917156e+10,2.398719e+12,2.110926e+12,10.665814,1.216131,2.877933e+11,10.136290,Grande
1,00.000.208/0001-00,BRB BANCO DE BRASILIA S.A.,2024-12-31 00:00:00.000000,8.909356e+09,3.110190e+08,6.250441e+10,5.391283e+10,3.490926,0.497595,8.591583e+09,3.620043,Grande
2,00.001.180/0001-26,CENTRAIS ELET BRAS S.A. - ELETROBRAS,2024-12-31 00:00:00.000000,4.018155e+10,1.038075e+10,2.898713e+11,1.678716e+11,25.834624,3.581159,1.219998e+11,8.508830,Grande
3,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2024-12-31 00:00:00.000000,3.503690e+08,2.255270e+08,1.484640e+09,1.695600e+08,64.368423,15.190686,1.315080e+09,17.149299,Grande
4,00.080.671/0001-00,CARAMURU ALIMENTOS S.A.,2024-12-31 00:00:00.000000,7.272804e+09,2.720760e+08,7.220957e+09,4.983928e+09,3.741006,3.767866,2.237029e+09,12.162381,Grande


Nota técnica: O uso do SQLAlchemy com context managers garante que as conexões sejam encerradas corretamente, evitando o travamento do banco de dados durante a análise.

#2. Análise de Negócio via SQL
Abaixo, realizamos consultas estratégicas para extrair insights sobre a saúde financeira e a eficiência das empresas listadas na CVM em 2024.

### 1: Qual o porte de empresa que entregou o melhor retorno (ROE) em 2024?
Objetivo: Identificar em qual faixa de tamanho as empresas estão sendo mais eficientes para o acionista.

In [6]:
query_1 = """
SELECT
    PORTE_EMPRESA,
    COUNT(*) as QTD_EMPRESAS,
    ROUND(AVG(ROE), 2) as ROE_MEDIO_PCT
FROM empresas_cvm
WHERE ROE IS NOT NULL
GROUP BY PORTE_EMPRESA
ORDER BY ROE_MEDIO_PCT DESC;
"""
df_porte = query_sql(query_1)
display(df_porte)

,PORTE_EMPRESA,QTD_EMPRESAS,ROE_MEDIO_PCT
0,Grande,342,0.99
1,Pequena,5,-0.36
2,Média,57,-77.66


Insight: Determina se a escala das empresas "Grandes" se traduz em maior rentabilidade ou se as "Médias/Pequenas" possuem operações mais enxutas e rentáveis.

### 2: Quais empresas operam com maior risco financeiro (Patrimônio Líquido Negativo)?
Objetivo: Listar empresas onde as dívidas totais superam os ativos (Passivo a Descoberto).

In [7]:
query_2 = """
SELECT
    DENOM_CIA,
    ROUND(PATRIMÔNIO_LÍQUIDO / 1e6, 2) as PL_MILHOES,
    PORTE_EMPRESA
FROM empresas_cvm
WHERE PATRIMÔNIO_LÍQUIDO < 0
ORDER BY PATRIMÔNIO_LÍQUIDO ASC
LIMIT 10;
"""
df_risco = query_sql(query_2)
display(df_risco)

,DENOM_CIA,PL_MILHOES,PORTE_EMPRESA
0,AZUL S.A.,-30435.27,Grande
1,GOL LINHAS AEREAS INTELIGENTES S.A.,-29090.52,Grande
2,COBRASMA S.A.,-27235.98,Média
3,REFINARIA DE PETROLEOS MANGUINHOS S.A.,-6512.16,Grande
4,BRASKEM S.A.,-4278.00,Grande
5,INVESTIMENTOS E PARTICIP. EM INFRA S.A. - INVEPAR,-4206.00,Grande
6,PDG REALTY S.A. EMPREEND E PARTICIPACOES,-3333.50,Média
7,UNIGEL PARTICIPAÇÕES S.A.,-3116.96,Grande
8,SANSUY S.A. INDUSTRIA DE PLASTICOS,-1985.20,Média
9,GENERAL SHOPPING E OUTLETS DO BRASIL BRASIL S.A.,-1533.92,Grande


Insight: Diferenças muito grandes (spread alto) indicam que o lucro é "alavancado" por dívida, o que aumenta o risco para o investidor em cenários de juros altos.

### 3: Quem são as 10 maiores geradoras de lucro absoluto no Brasil?
Objetivo: Rankear as empresas que dominam o mercado em volume financeiro de lucro.

In [8]:
query_3 = """
SELECT
    DENOM_CIA,
    ROUND(LUCRO_LÍQUIDO / 1e6, 2) as LUCRO_MILHOES,
    ROUND(MARGEM_LÍQUIDA, 2) as MARGEM_PCT
FROM empresas_cvm
ORDER BY LUCRO_LÍQUIDO DESC
LIMIT 10;
"""
df_top_lucro = query_sql(query_3)
display(df_top_lucro)

,DENOM_CIA,LUCRO_MILHOES,MARGEM_PCT
0,PETROLEO BRASILEIRO S.A. PETROBRAS,37009.00,7.54
1,VALE S.A.,30431.00,14.77
2,BCO BRASIL S.A.,29171.56,10.67
3,BCO BRADESCO S.A.,17542.15,8.23
4,ITAÚSA S.A.,14887.00,180.78
5,AMBEV S.A.,14846.95,16.60
6,BCO SANTANDER (BRASIL) S.A.,13413.76,9.78
7,JBS S.A.,10703.99,2.57
8,CENTRAIS ELET BRAS S.A. - ELETROBRAS,10380.75,25.83
9,PRIO S.A.,10301.61,71.73


Insight: Revela a concentração de lucros e permite avaliar se as maiores empresas em volume também mantêm margens saudáveis.

###4: Qual o nível de alavancagem das empresas mais rentáveis?
Objetivo: Verificar se o ROE alto é fruto de eficiência (ROA) ou de excesso de endividamento (Spread).

In [9]:
query_4 = """
SELECT
    DENOM_CIA,
    ROUND(ROE, 2) as ROE_PCT,
    ROUND(ROA, 2) as ROA_PCT,
    ROUND(ROE - ROA, 2) as SPREAD_ALAVANCAGEM
FROM empresas_cvm
WHERE PORTE_EMPRESA = 'Grande'
  AND ROA > 0
  AND ROE > 0
ORDER BY SPREAD_ALAVANCAGEM DESC
LIMIT 10;
"""
df_alavancagem_grande = query_sql(query_4)
display(df_alavancagem_grande)

,DENOM_CIA,ROE_PCT,ROA_PCT,SPREAD_ALAVANCAGEM
0,SANTOS BRASIL PARTICIPACOES S.A.,112.19,13.39,98.80
1,CONCESSIONÁRIA DA LINHA 4 DO METRÔ DE SÃO PAUL...,72.22,11.40,60.82
2,MAHLE METAL LEVE S.A.,70.23,15.06,55.17
3,MADERO INDÚSTRIA E COMÉRCIO S.A.,61.42,7.82,53.60
4,TECIDOS E ARMARINHOS MIGUEL BARTOLOMEU S.A.,59.11,9.14,49.97
5,BB SEGURIDADE PARTICIPAÇÕES S.A.,89.77,40.26,49.50
6,RODOVIAS DO BRASIL HOLDING S.A.,48.26,4.13,44.12
7,ALUBAR METAIS E CABOS S.A.,44.62,5.04,39.58
8,CURY CONSTRUTORA E INCORPORADORA S.A.,53.36,16.10,37.26
9,EPR INFRAESTRUTURA PR S.A.,60.74,27.83,32.91


Insight: Diferenças muito grandes (spread alto) indicam que o lucro é "alavancado" por dívida, o que aumenta o risco para o investidor em cenários de juros altos.

###5: Quais empresas faturam muito mas operam com margem negativa?
Objetivo: Detectar ineficiência operacional: empresas com alta receita que não conseguem gerar lucro.

In [10]:
query_5 = """
SELECT
    DENOM_CIA,
    ROUND(RECEITA_VENDA / 1e6, 2) as RECEITA_MILHOES,
    ROUND(MARGEM_LÍQUIDA, 2) as MARGEM_PCT
FROM empresas_cvm
WHERE RECEITA_VENDA > 5e8 -- Filtro: Receita > 500 Milhões
  AND MARGEM_LÍQUIDA < 0
ORDER BY MARGEM_LÍQUIDA ASC;
"""
df_alerta = query_sql(query_5)
display(df_alerta)

,DENOM_CIA,RECEITA_MILHOES,MARGEM_PCT
0,SEQUOIA LOGÍSTICA E TRANSPORTES S.A.,769.02,-166.24
1,INFRACOMMERCE CXAAS S.A.,1065.32,-164.85
2,BRASIL BIOFUELS S.A.,608.45,-73.68
3,AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. ...,1516.48,-61.60
4,UNIGEL PARTICIPAÇÕES S.A.,3148.94,-59.15
...,...,...,...
68,COMPANHIA BRASILEIRA DE ALUMÍNIO,8173.65,-0.89
69,AUTOMOB PARTICIPAÇÕES S.A.,12240.03,-0.86
70,"ALLPARK EMPREENDIMENTOS, PARTICIPAÇÕES E SERVI...",1584.81,-0.55
71,YOU INC INCORPORADORA E PARTICIPAÇÃO S.A,1139.57,-0.38


Insight: Expõe empresas que, apesar do grande volume de vendas, possuem estruturas de custos que consomem todo o faturamento, resultando em prejuízo operacional.